In [ ]:
from dotenv import load_dotenv
load_dotenv()

%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
# %matplotlib inline
# %matplotlib qt5
# import mne
# mne.viz.set_browser_backend("qt")  # or "matplotlib"
# mne.set_config("MNE_BROWSER_BACKEND", "qt")  # or "matplotlib"
%gui qt

from copy import deepcopy
import time
import xarray as xr # Assuming you're using this
import numpy as np   # For the example
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import xarray as xr
# import zarr
import panel as pn
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
# import holoviews as hv
# hv.extension('bokeh', logo=False)

# import hvplot.xarray
# import hvplot.pandas
# This line is crucial for displaying plots in a notebook
# hvplot.extension('bokeh') # You can also use 'matplotlib' or 'plotly'

# hv.extension('bokeh')
# hv.extension('matplotlib') # or 'matplotlib'
# hv.extension('plotly') # or 'matplotlib'
# from holoviews import opts
import panel as pn
pn.extension()

import IPython

# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pyqtgraph as pg
from phopymnehelper.historical_data import HistoricalData
from pypho_timeline.widgets import SimpleTimelineWidget
from pypho_timeline.__main__ import VideoTrackDatasource

from pypho_timeline.timeline_builder import TimelineBuilder

def get_now_time_str(time_separator='-') -> str:
    return str(time.strftime(f"%Y-%m-%d_%H{time_separator}%m", time.localtime(time.time())))



# Create Qt application
app = pg.mkQApp("pyPhoTimelineTestingNotebookExample")

builder: TimelineBuilder = TimelineBuilder()


### Motion track: BAD intervals (exclude + dark overlay)

`MotionTrackDatasource` accepts optional bad segments aligned with `motion_df['t']` (Unix seconds or datetimes in the same time base as the track).

- **`bad_intervals_df`**: either columns `t_start` + `t_duration`, or MNE-style `onset` + `duration` (then pass **`bad_intervals_time_origin_unix`** = recording start as Unix seconds so `t_start = origin + onset`).
- **`exclude_bad_from_detail`** (default `True`): rows whose time falls inside a bad interval are dropped **before** downsampling.
- **`bad_overlay_alpha`** (default `0.9`): vertical black bands drawn on top of the motion lines via `LinearRegionItem`.

Helpers: `normalize_motion_bad_intervals_df`, `motion_bad_intervals_key_suffix`.

**PhoPyMNEHelper `is_moving_annots_df`** (has `onset`, `duration` in seconds relative to the recording):

```python
from pypho_timeline.rendering.datasources.specific.motion import MotionTrackDatasource, normalize_motion_bad_intervals_df

recording_start_unix = float(your_meas_date.timestamp())  # same origin as timeline motion `t`
ds = MotionTrackDatasource(
    intervals_df, motion_df,
    bad_intervals_df=is_moving_annots_df,
    bad_intervals_time_origin_unix=recording_start_unix,
    exclude_bad_from_detail=True,
    bad_overlay_alpha=0.9,
)
```

If bad intervals are already in absolute Unix time, use `bad_intervals_df=normalize_motion_bad_intervals_df(df)` and omit `bad_intervals_time_origin_unix`.

After **`set_bad_intervals(...)`** on an already displayed track, refresh the cached renderer if needed: `track_renderer.detail_renderer = ds.get_detail_renderer()` (and re-trigger a viewport refresh / overview update as your app does for `source_data_changed_signal`).

In [ ]:
# n_most_recent_sessions_to_preprocess: int = None # None means all sessions
# n_most_recent_sessions_to_preprocess: int = 100
# n_most_recent_sessions_to_preprocess: int = 15
n_most_recent_sessions_to_preprocess: int = 5
# n_most_recent_sessions_to_preprocess: int = 3
# n_most_recent_sessions_to_preprocess = None

# Optional: only include streams whose name matches any of these patterns (regex).
STREAM_ALLOWLIST: Optional[List[str]] = None  # e.g. [r"EEG.*", r"MOTION.*"]

# Optional: exclude streams whose name matches any of these patterns (regex).
# STREAM_BLOCKLIST: Optional[List[str]] = ['Epoc X Motion', 'Epoc X eQuality']
STREAM_BLOCKLIST: Optional[List[str]] = ['Epoc X eQuality']

# onOptionsAccepted

# db_root_path = Path('/content/drive/MyDrive/Databases')
# db_root_path = Path(r'E:/Dropbox (Personal)/Databases') ## APOGEE
db_root_path = Path(r'E:/Dropbox (Personal)/Databases') # WIN10_VM
assert db_root_path.exists(), f"'{db_root_path.as_posix()}' does not exist!"

# eeg_recordings_file_path: Path = Path(r'E:/Dropbox (Personal)/Databases/UnparsedData/EmotivEpocX_EEGRecordings/fif')
# headset_motion_recordings_file_path: Path = Path(r'E:/Dropbox (Personal)/Databases/UnparsedData/EmotivEpocX_EEGRecordings/MOTION_RECORDINGS/fif')

# assert eeg_recordings_file_path.exists()
# assert headset_motion_recordings_file_path.exists()

# eeg_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEpocX_EEGRecordings/fif')
# flutter_eeg_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings')
# flutter_motion_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings/MOTION_RECORDINGS')
# flutter_GENERIC_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings/GENERIC_RECORDINGS')

# headset_motion_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEpocX_EEGRecordings/MOTION_RECORDINGS/fif')
# WhisperVideoTranscripts_LSL_Converted = db_root_path.joinpath('UnparsedData/WhisperVideoTranscripts_LSL_Converted')
pho_log_to_LSL_recordings_path: Path = db_root_path.joinpath('UnparsedData/PhoLogToLabStreamingLayer_logs')
## These contain little LSL .fif files with names like: '20250808_062814_log.fif',

# eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed')
# pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION')
# assert pickled_data_path.exists()
xdf_to_rerun_rrd_parent_export_path = db_root_path.joinpath('AnalysisData/to_rerun_rrd').resolve()
xdf_to_rerun_rrd_parent_export_path.mkdir(exist_ok=True)
# print(f'xdf_to_rerun_rrd_parent_export_path: "{xdf_to_rerun_rrd_parent_export_path.as_posix()}"')

# lab_recorder_output_path = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001")
lab_recorder_output_path = db_root_path.joinpath('UnparsedData/LabRecorderStudies/sub-P001')
assert lab_recorder_output_path.exists()



# modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=lab_recorder_output_path, recordings_extensions=['.xdf'])
modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], recordings_extensions=['.xdf']) ## both sources
# modern_found_EEG_recording_files

most_recent_modern_found_EEG_recording_files: List[Path] = modern_found_EEG_recording_files[:n_most_recent_sessions_to_preprocess]
# most_recent_modern_found_EEG_recording_files


# active_EEG_recording_files = modern_found_EEG_recording_files
active_EEG_recording_files = most_recent_modern_found_EEG_recording_files

print(f'processing len(active_EEG_recording_files): {len(active_EEG_recording_files)} recording files...')
most_recent_modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=active_EEG_recording_files) ## 8m for 65 files
# most_recent_modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=most_recent_modern_found_EEG_recording_files) ## 5m for 35 files !! SLOWER: 9.2min for 35 files
most_recent_modern_found_EEG_recording_file_df


# modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=modern_found_EEG_recording_files)
# modern_found_EEG_recording_file_df

## OUTPUTS: modern_found_EEG_recording_file_df, modern_found_EEG_recording_files

## INPUTS: most_recent_modern_found_EEG_recording_file_df

xdf_file_cache_filename: str = f"{get_now_time_str()}_found_xdf_files.csv"
xdf_file_cache_filepath: Path = xdf_to_rerun_rrd_parent_export_path.joinpath(xdf_file_cache_filename).resolve()

print(f'exporting xdf .csv to xdf_file_cache_filepath: "{xdf_file_cache_filepath}..."')
most_recent_modern_found_EEG_recording_file_df.to_csv(xdf_file_cache_filepath)


most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()
final_xdf_paths: List[Path] = [Path(v) for v in most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()]
final_xdf_paths

## OUTPUTS: final_xdf_paths



# active_xdf_path: List[Path] = [
#     # Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-05T022422.773Z_eeg.xdf").resolve(),
#     # Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-12-24T213732.785Z_eeg.xdf").resolve(),
#     # Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-05T022422.773Z_eeg.xdf").resolve(),
#     # Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-09T233324.449Z_eeg.xdf").resolve(), ## COGNITION WAS ABYSMAL, most sleep debt in a single week, only 3-day-binge.

#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T222456.756Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T222210.952Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T221511.412Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T194023.711Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T134325.495Z_eeg.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z.xdf"),
#     # Path("E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T023809.607Z_eeg.xdf"),

#     # # 2026-03-01 _________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________ #
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-01T021033.491Z_eeg.xdf'),
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260301_020924_log.xdf'),
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260301_015939_log.xdf'),
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-02-28T152439.698Z_eeg.xdf'),
#     # Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260228_152426_log.xdf'),

#     # 2026-03-04 _________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________ #
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-04T225210.949Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260304_225204_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-04T192035.507Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260304_192023_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-04T191528.965Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260304_191511_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-04T162633.469Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260304_162623_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-03T223004.805Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260303_222952_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-03T191723.860Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260303_191710_log.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-03T012438.863Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-03-03T005759.073Z_eeg.xdf'),
#     Path('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260303_005724_log.xdf'),
    
# ]


active_xdf_path: List[Path] = final_xdf_paths


## INPUTS: active_xdf_path
timeline: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=active_xdf_path,
    # stream_allowlist=[r"EEG.*", r"MOTION.*"],
    #  stream_allowlist=[r'Epoc X*', r'Epoc X Motion*'], # , 'WhisperLiveLogger', 'Epoc X eQuality', 'TextLogger', 'EventBoard'
    # stream_allowlist=[r'Epoc X', r'Epoc X Motion*'], # , 'WhisperLiveLogger', 'Epoc X eQuality', 'TextLogger', 'EventBoard'
    # stream_blocklist=['WhisperLiveLogger', 'Epoc X eQuality', 'TextLogger', 'EventBoard']
    stream_blocklist=STREAM_BLOCKLIST,
)
# assert demo_xdf_path.exists()


======== STREAM "EventBoard":
	WARN: skipping empty stream: "EventBoard"
======== STREAM "Epoc X":
	created_at_dt: 2026-03-27 10:51:34 PM
	first_timestamp_dt: 2026-03-27 10:51:34 PM
	last_timestamp_dt: 2026-03-27 10:51:34 PM
WARN: (len(phopylslhelper_dict)3
	 FOUND CUSTOM TIMESTAMP SYNC KEY: "stream_start_datetime": 2026-03-27 22:51:13+00:00
	 FOUND CUSTOM TIMESTAMP SYNC KEY: "stream_start_lsl_local_offset_seconds": 875048.424857


In [ ]:
timeline.get_all_track_names()


In [ ]:
a_widget, a_renderer, a_ds = timeline.get_track_tuple('LOG_TextLogger')

In [ ]:
rendered_items = list(a_renderer.detail_graphics.values())[0]

rendered_text_items: list[pg.TextItem] = [v for v in rendered_items if isinstance(v, pg.TextItem)]
rendered_text_items


In [ ]:
for a_text_item in rendered_text_items:
    a_text_item.setAnchor([0.5, 0.5])
    a_text_item.set

In [ ]:
from pypho_timeline.rendering.datasources.specific import EEGTrackDatasource


a_ds: EEGTrackDatasource = timeline.track_datasources['EEG_Epoc X']
a_ds

In [ ]:
a_ds.intervals_df

In [ ]:

timeline = builder.build_from_xdf_file(demo_xdf_path)


In [ ]:
# timeline = main_all_modalities_from_xdf_file_example(demo_xdf_path)

# Motion Bad Period Detection

In [ ]:
timeline.get_all_track_names()

In [ ]:
eeg_spectogram_track_names = ['EEG_Spectrogram_Epoc X_Frontal-L',
 'EEG_Spectrogram_Epoc X_Frontal-R',
 'EEG_Spectrogram_Epoc X_Posterior-L',
 'EEG_Spectrogram_Epoc X_Posterior-R',]

for a_spectogram_track_name in eeg_spectogram_track_names:
    a_spectogram_widget, a_spectogram_track, a_spectogram_ds = timeline.get_track_tuple(a_spectogram_track_name)
    # a_spectogram_track
    a_root_graphics_layout_widget = a_spectogram_widget.getRootGraphicsLayoutWidget()
    a_plot_item = a_spectogram_widget.getRootPlotItem()
    # a_plot_item
    a_spectogram_ds.detailed_df

In [ ]:
motion_track = timeline.get_track('MOTION_Epoc X Motion')
motion_track

In [ ]:
from phopymnehelper.motion_data import MotionData, MotionDataFrame, BadMotionDataFrame
from phopymnehelper.MNE_helpers import MNEHelpers

motion_widget, motion_track, motion_ds = timeline.get_track_tuple('MOTION_Epoc X Motion')


In [ ]:
total_accel_threshold: float = 0.5

# a_motion_df: pd.DataFrame = motion_ds.detailed_df.copy() # a_ds.to_data_frame()       
# a_motion_df: pd.DataFrame = MotionData.compute_rolling_motion_change_detection(a_df=a_motion_df, total_change_threshold=total_accel_threshold, enable_global_normalization=True)
is_moving_annots, is_moving_annots_df = MotionData.find_high_accel_periods(a_ds=motion_ds.detailed_df, total_change_threshold=total_accel_threshold, 
    should_set_bad_period_annotations=False, minimum_bad_duration=0.050) # at least 50ms in duration to prevent tons of tiny intervals

motion_ds.detailed_df


In [ ]:


is_moving_annots_df.bad_motion_epochs_df.is_timestamp_format()


In [ ]:
## How can I mask an existing track (such as the Motion Track) using a set of BAD_INTERVALS like is_moving_annots_df: pd.DataFrame? I'd like the data object to be set to indicate that these periods should be excluded, and the BAD periods to be rendered darkened by overlaying a BLACK fill with 90% opacity for the regions
# motion_track.set_bad_intervals(bad_intervals_df=is_moving_annots_df)
motion_ds.set_bad_intervals(bad_intervals_df=is_moving_annots_df)

In [ ]:
from pypho_timeline._embed.interval_datasource import IntervalsDatasource
from pypho_timeline.rendering.graphics.interval_rects_item import IntervalRectsItem, IntervalRectsItemData
from pypho_timeline.rendering.helpers.render_rectangles_helper import Render2DEventRectanglesHelper

# motion_ds._bad_intervals_unix_df
# _detail_t_column_to_unix_numpy
# motion_ds._bad_intervals_unix_df['t_start'] = motion_ds._bad_intervals_unix_df['onset']

motion_ds._bad_intervals_unix_df = motion_ds._bad_intervals_unix_df.bad_motion_epochs_df.adding_unix_float_columns(inplace=False)
# motion_ds.bad_intervals_unix_df
print(motion_ds._bad_intervals_unix_df.dtypes)
# onset           datetime64[ns]
# duration       timedelta64[ns]
# description             object
# t_start                float64
# t_duration             float64


In [ ]:
motion_ds._bad_intervals_unix_df

In [ ]:
bad_motion_intervals_ds: IntervalsDatasource = IntervalsDatasource(df=motion_ds._bad_intervals_unix_df, datasource_name='bad_motion_epochs')
bad_motion_intervals_ds


In [ ]:
motion_widget._update_plots()

In [ ]:
# motion_widget
_out_bad_motion_rendered_intervals = timeline.add_rendered_intervals(interval_datasource=motion_ds._bad_intervals_unix_df, name='bad_motion_epochs',
                                                child_plots=[a_plot_item],
                                                debug_print=True)

# motion_track.add_render

In [ ]:
plot_item = _out_bad_motion_rendered_intervals['RootPlot']['plot']
rect_item: IntervalRectsItem = _out_bad_motion_rendered_intervals['RootPlot']['rect_item']
rect_item

In [ ]:
plot_item.zValue()
rect_item.zValue()
# plot_item.children()

rect_item.setZValue(50)


In [ ]:
an_IntervalRectsItem: IntervalRectsItem = Render2DEventRectanglesHelper.build_IntervalRectsItem_from_interval_datasource(interval_datasource=bad_motion_intervals_ds)

motion_

In [ ]:
motion_track.

In [ ]:
timeline.hide_extra_xaxis_labels_and_axes()

In [ ]:
motion_track.visible_intervals

## End motion annotations

In [ ]:
hasattr(motion_track, "channels_enabled")
motion_track.update_channel_visibility(channel_name='GyroX', is_visible=False)
motion_track.update_channel_visibility(channel_name='GyroY', is_visible=False)
motion_track.update_channel_visibility(channel_name='GyroZ', is_visible=False)

In [ ]:
motion_detail_renderer = motion_track.detail_renderer
motion_detail_renderer.normalize = False

motion_track.datasource.detailed_df
# motion_detail_renderer
# motion_detail_renderer.motion_df


In [ ]:
## INPUTS: timeline
# Get the "EEG Epoc X" track from the timeline
# Fallback to searching tracks by attribute, as there is no `.get_track_by_name` method
# eeg_track = timeline.get_track('EEG_Epoc X')
eeg_widget, eeg_track, eeg_ds = timeline.get_track_tuple('EEG_Epoc X')
detailed_eeg_df: pd.DataFrame = eeg_ds.detailed_df
# print(detailed_eeg_df.columns) # ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4', 't']
detailed_eeg_df['t']




In [ ]:
mask_bad_intervals_df: pd.DataFrame = motion_ds._bad_intervals_unix_df.copy()
mask_bad_intervals_df['onset_end'] = mask_bad_intervals_df['onset'] + mask_bad_intervals_df['duration']
mask_bad_intervals_df

In [ ]:
from phopymnehelper.helpers.dataframe_accessor_helpers import MaskedValidDataFrameAccessor

# mask_by_intervals

## find if each detailed_eeg_df['t'] is within the bad interval
# detailed_eeg_df['t']
# Efficiently see if each `detailed_eeg_df['t']` value falls within `mask_bad_intervals_df[['onset', 'onset_end']]` where all columns are datetime64[ns] 

# mask_bad_intervals_df[['onset', 'onset_end']]
# detailed_eeg_df['t'].dtypes # dtype('<M8[ns]')

detailed_eeg_df = detailed_eeg_df.masked_df.masking_by_intervals(mask_bad_intervals_df=mask_bad_intervals_df, time_col_name='t', bool_mask_column_name='is_bad_motion',
                                                            intervals_start_col_name='onset', intervals_end_col_name='onset_end')
detailed_eeg_df.masked_df.add_masking_column(mask_col='is_valid', value_cols=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4'])
detailed_eeg_df


In [ ]:
detailed_eeg_df

In [ ]:


masked_detailed_eeg_df: pd.DataFrame = detailed_eeg_df.masked_df.get_masked(copy=True)

# masked_detailed_eeg_df: pd.DataFrame = detailed_eeg_df.masked_df.add_masking_column(mask_col='is_valid', value_cols=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4'],
#                                                      copy=True)
masked_detailed_eeg_df

In [ ]:
masked_detailed_eeg_df.masked_df.get_masked()

In [ ]:
detailed_eeg_df = detailed_eeg_df.drop(columns=['is_bad_motion_right', 'is_bad_motion'], inplace=False)

In [ ]:

# ii = pd.IntervalIndex.from_arrays(
#     mask_bad_intervals_df["onset"],
#     mask_bad_intervals_df["onset_end"],
#     closed="both",  # or "left" / "right" / "neither"
# )
# in_bad = ii.contains(detailed_eeg_df["t"].values)


# import polars as pl

# points = pl.from_pandas(detailed_eeg_df).lazy()  # or scan_csv / DataFrame
# intervals = pl.from_pandas(mask_bad_intervals_df).select("onset", "onset_end").lazy()

# hits = points.join_where(
#     intervals,
#     pl.col("t").is_between(pl.col("onset"), pl.col("onset_end")),
# )

# # timestamps that fall in at least one bad interval
# in_bad = (
#     hits.select("t")
#     .unique()
#     .with_columns(in_bad=pl.lit(True))
# )

# out = (
#     points.join(in_bad, on="t", how="left")
#     .with_columns(pl.col("in_bad").fill_null(False))
# )

# (pl.col("t") >= pl.col("onset")) & (pl.col("t") < pl.col("onset_end"))

import polars as pl

lf = pl.from_pandas(detailed_eeg_df).lazy()
iv = pl.from_pandas(mask_bad_intervals_df).select("onset", "onset_end").lazy()

bad_t = (
    lf.join(iv, how="cross")
    .filter((pl.col("t") >= pl.col("onset")) & (pl.col("t") <= pl.col("onset_end")))
    .select("t")
    .unique()
    .with_columns(pl.lit(True).alias("is_bad_motion"))
)

df = lf.join(bad_t, on="t", how="left").with_columns(
    pl.col("is_bad_motion").fill_null(False)
).collect()

detailed_eeg_df = df.to_pandas()  # if you need pandas again
detailed_eeg_df


In [ ]:
# eeg_track = timeline.get_track('MOTION_Epoc X Motion')

# if hasattr(timeline, "tracks"):
#     for tr in timeline.tracks:
#         # Try 'name' or 'track_name' attribute, fallback to 'datasource.name'
#         track_name = getattr(tr, "name", None) or getattr(tr, "track_name", None)
#         if track_name == "EEG Epoc X":
#             eeg_track = tr
#             break
#         elif hasattr(tr, "datasource") and hasattr(tr.datasource, "name"):
#             if tr.datasource.name == "EEG Epoc X":
#                 eeg_track = tr
#                 break
# eeg_track
eeg_track.channel_visibility

In [ ]:
hasattr(eeg_track, "channels_enabled")
eeg_track.update_channel_visibility(channel_name='GyroX', is_visible=False)


In [ ]:
channels_to_hide = [] # ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
channels_to_hide = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'FC6', 'F4', 'F8', 'AF4'] # , 'O1', 'O2', 'P8', 'T8'
for a_ch in channels_to_hide:
    eeg_track.update_channel_visibility(channel_name=a_ch, is_visible=False)
# eeg_track.update_channel_visibility(channel_name='P8', is_visible=False)


In [ ]:
print(list(eeg_track.channel_visibility.keys()))

In [ ]:
eeg_track.visible_intervals ## this should not be empty


In [ ]:


eeg_track._trigger_visibility_render()

In [ ]:

if eeg_track is not None and hasattr(eeg_track, "channel_visibility"):
    # If there's an attribute listing enabled channels, we'll disable half
    enabled_channels = eeg_track.channel_visibility
    # Otherwise fall back to all channel names if present
    if not enabled_channels and hasattr(eeg_track, "all_channel_names"):
        enabled_channels = eeg_track.all_channel_names
    if enabled_channels:
        # Compute half the channels (disable the second half)
        half = len(enabled_channels) // 2
        # Set only the first half as enabled
        eeg_track.channel_visibility = enabled_channels[:half]
        # If the track supports setting enabled/disabled channels via a method, use it
        if hasattr(eeg_track, 'set_enabled_channels'):
            eeg_track.set_enabled_channels(enabled_channels[:half])
        # Potentially trigger an update/redraw
        if hasattr(eeg_track, 'update_channel_visibility'):
            eeg_track.update_channel_visibility()
else:
    print("Could not find 'EEG Epoc X' track or it doesn't support channel control.")


# Timeline Widget from just Video Track

In [ ]:
from pathlib import Path
import os

# Specify your video folder path
# video_folder = Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/Videos")
video_folder = Path(r"M:/ScreenRecordings/EyeTrackerVR_Recordings")
assert video_folder.exists() and video_folder.is_dir()

# Gather all video files (adjust extensions as needed)
video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.wmv')
all_videos = [p for p in video_folder.glob('*') if p.suffix.lower() in video_extensions]

# Sort by modification time (descending), get 5 most recent
all_videos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
recent_videos = all_videos[:10]
recent_videos

# Create the VideoTrackDatasource
video_ds: VideoTrackDatasource = VideoTrackDatasource(video_paths=recent_videos)

# Choose a name for the new video track
video_track_name: str = "RecentVideosTrack"



In [ ]:

# Add the new video track to the existing timeline
# timeline.add_video_track(video_track_name, video_ds)
video_widget, root_graphics, plot_item, dock = timeline.add_video_track(track_name=video_track_name, video_datasource=video_ds,
                                                                                                                enable_time_crosshair=True,
                                                                                                            )

# timeline.add_track(video_ds, name=video_track_name)


In [ ]:
# Build a new timeline from just video track (no XDF file needed)
# Option 1: From video folder
# video_timeline = builder.build_from_video(video_folder_path=video_folder)

# Option 2: From list of video paths
video_timeline = builder.build_from_video(video_paths=recent_videos)

# Option 3: From existing VideoTrackDatasource
# video_timeline = builder.build_from_video(video_datasource=video_ds)
video_timeline.show()

In [ ]:
video_timeline.get_all_track_names()

In [ ]:
# video_widget
video_track_name = "VideoTrack"
video_widget, video_track_renderer, video_track_datasource = video_timeline.get_track_tuple(name=video_track_name)
video_widget

In [ ]:
## Add a current datetime red line representing the current datetime (datetime.now())

video_widget.


In [ ]:
video_timeline.show()

In [ ]:
# Create a plot widget for this track
from pypho_timeline.core.synchronized_plot_mode import SynchronizedPlotMode


a_detail_renderer = video_ds.get_detail_renderer()

# The getOptionsPanel() method will be called by the dock when needed
# No need to set optionsPanel attribute if getOptionsPanel() is implemented

## if we provide a valid `optionsPanel: Optional[QWidget]` here on the widget the button will automatically show up on the dock
track_widget, a_root_graphics, a_plot_item, a_dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
    name=video_ds.custom_datasource_name,
    dockSize=(500, 80),
    dockAddLocationOpts=['bottom'],
    sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA
)

assert a_detail_renderer is not None
track_widget.set_track_renderer(a_detail_renderer)
# Explicitly set the attribute (not just rely on getOptionsPanel())
track_widget.optionsPanel = track_widget.getOptionsPanel()

# Try to force dock to update button visibility
a_dock.updateWidgetsHaveOptionsPanel()

a_dock.update()  # May refresh the title bar
# Or if available:
if hasattr(a_dock, 'updateTitleBar') or hasattr(a_dock, 'refresh'):
    a_dock.updateTitleBar()  # or refresh()



# Set the plot to show the full time range
a_plot_item.setXRange(
    timeline.total_data_start_time, 
    timeline.total_data_end_time, 
    padding=0
)
a_plot_item.setYRange(0, 1, padding=0)
a_plot_item.setLabel('bottom', 'Time', units='s')
a_plot_item.setLabel('left', video_ds.custom_datasource_name)
a_plot_item.hideAxis('left')  # Hide Y-axis for cleaner look

# Add the track to the plot
a_track_name: str = video_ds.custom_datasource_name
timeline.add_track(
    video_ds,
    name=a_track_name,
    plot_item=a_plot_item
)

# Messing with stream:

In [ ]:
a_track_name: str = 'MOTION_Epoc X Motion'
# a_track_name: str = 'EEG_Epoc X'
a_renderer = timeline.track_renderers[a_track_name]
a_detail_renderer = a_renderer.detail_renderer # MotionPlotDetailRenderer 
a_ds = timeline.track_datasources[a_track_name]
# interval = a_ds.intervals_df
interval = a_ds.get_overview_intervals()
# interval = None
type(interval)
interval

In [ ]:
a_renderer.visible_intervals
a_renderer.detail_graphics
a_renderer.overview_rects_item



In [ ]:
from pypho_timeline.core.pyqtgraph_time_synchronized_widget import PyqtgraphTimeSynchronizedWidget

dDisplayItem = timeline.ui.dynamic_docked_widget_container.find_display_dock(identifier=a_track_name) # Dock
a_widget: PyqtgraphTimeSynchronizedWidget = timeline.ui.matplotlib_view_widgets[a_track_name] # PyqtgraphTimeSynchronizedWidget 
a_root_graphics_layout_widget = a_widget.getRootGraphicsLayoutWidget()
a_plot_item = a_widget.getRootPlotItem()
a_plot_item


In [ ]:
# Option 2: directly on the PlotItem (delegates to its ViewBox)
(x_min, x_max), (y_min, y_max) = a_plot_item.viewRange()

# The currently visible time window on the track:
t_start, t_end = x_min, x_max

active_viewport_duration_t: float = t_end - t_start
active_viewport_duration_t
(t_start, t_end) # (59086.148423519495, 59102.162193802214)
interval_dict_rep = {'t_start': t_start, 't_duration': active_viewport_duration_t, 't_end': t_end}
interval_dict_rep

In [ ]:
def getOptionsPanel(self):
    """Return the options panel widget for this dock item."""
    from PyQt5.QtWidgets import QWidget, QVBoxLayout, QLabel, QCheckBox, QSpinBox, QGroupBox
    
    options_widget = QWidget()
    main_layout = QVBoxLayout()
    
    # Create a group box for organization
    settings_group = QGroupBox("Settings")
    settings_layout = QVBoxLayout()
    
    # Add your actual settings controls
    self.option_checkbox = QCheckBox("Enable feature")
    settings_layout.addWidget(self.option_checkbox)
    
    self.option_spinbox = QSpinBox()
    self.option_spinbox.setRange(1, 100)
    settings_layout.addWidget(QLabel("Value:"))
    settings_layout.addWidget(self.option_spinbox)
    
    settings_group.setLayout(settings_layout)
    main_layout.addWidget(settings_group)
    main_layout.addStretch()
    
    options_widget.setLayout(main_layout)
    return options_widget

In [ ]:
a_widget._update_plots()

In [ ]:
# a_widget.options_panel
an_options_panel = a_widget.getOptionsPanel()
a_widget.optionsPanel = an_options_panel
a_widget.options_panel.show()




In [ ]:
# def _checkWidgetForOptions(self, widget):
#     """Check if a widget provides an options panel.
    
#     Returns:
#         Optional[QtWidgets.QWidget]: The options panel widget if found, None otherwise.
        
#     A widget can provide options in two ways:
#     1. Implement a method: getOptionsPanel() -> Optional[QWidget]
#     2. Provide an attribute: optionsPanel: Optional[QWidget]
#     """
#     # Check for method
#     if hasattr(widget, 'getOptionsPanel') and callable(getattr(widget, 'getOptionsPanel')):
#         try:
#             options_panel = widget.getOptionsPanel()
#             if options_panel is not None and isinstance(options_panel, QtWidgets.QWidget):
#                 return options_panel
#         except Exception as e:
#             # Silently ignore errors in getOptionsPanel
#             pass
#     # Check for attribute
#     if hasattr(widget, 'optionsPanel'):
#         options_panel = widget.optionsPanel
#         if options_panel is not None and isinstance(options_panel, QtWidgets.QWidget):
#             return options_panel
#     return None



In [ ]:
# Create the options panel
a_widget.optionsPanel = QWidget()
layout = QVBoxLayout()

# Add your options controls here
label = QLabel("Options for this widget")
layout.addWidget(label)

a_widget.optionsPanel.setLayout(layout)


In [ ]:
a_ds.detailed_df
a_ds.enable_downsampling
a_ds.max_points_per_second

a_ds.max_points_per_second


In [ ]:
a_ds.fetch_detailed_data(interval=interval_dict_rep)

In [ ]:
a_widget

In [ ]:
a_widget.active_plot_target


In [ ]:
a_ds.detailed_df

In [ ]:
graphics_objects = a_detail_renderer.render_detail(plot_item=a_plot_item, interval=interval, detail_data=a_ds.detailed_df) # List[PlotDataItem]
graphics_objects

In [ ]:
# clear_detail
a_detail_renderer.clear_detail(plot_item=a_plot_item, graphics_objects=graphics_objects)
a_detail_renderer

In [ ]:
# a_bounds_tuple = a_detail_renderer.get_detail_bounds(interval=a_ds.get_overview_intervals(), detail_data=a_ds.detailed_df) # List[PlotDataItem]
a_bounds_tuple = a_detail_renderer.get_detail_bounds(interval=interval, detail_data=a_ds.detailed_df) # List[PlotDataItem]
print(f'a_bounds_tuple: {a_bounds_tuple}')
x_min, x_max, y_min, y_max = a_bounds_tuple

In [ ]:
timeline.spikes_window.active_time_window
timeline.spikes_window.total_df_start_end_times


In [ ]:
# In your notebook, check:
track_name = 'MOTION_Epoc X Motion'
proxy_key = f'track_viewport_{track_name}'
print(f"Connection exists: {proxy_key in timeline.ui.connections}")

In [ ]:
# Check if sigRangeChanged has connections
viewbox = a_plot_item.getViewBox()
if viewbox is not None:
    print(f"sigRangeChanged connections: {viewbox.sigRangeChanged.receivers}")

In [ ]:
# Get the current viewport and trigger update
viewbox = a_plot_item.getViewBox()
if viewbox is not None:
    x_range, y_range = viewbox.viewRange()
    a_renderer.update_viewport(x_range[0], x_range[1])

## Individual parts of load from xdf file

In [ ]:
xdf_file_path = demo_xdf_path
print("=" * 60)
print(f"pyPhoTimeline - Load all EEG (or LSL) modalities from XDF: {xdf_file_path}")
print("=" * 60)

# Create Qt application
app = pg.mkQApp("pyPhoTimelineXDFExample")

# --- 1. Load the XDF file (using pyxdf) ---
import pyxdf

print(f"Loading XDF file: {xdf_file_path} ...")
streams, file_header = pyxdf.load_xdf(str(xdf_file_path))
print(f"Streams loaded: {[s['info']['name'][0] for s in streams]}")
print(f"File Header: {file_header}")

In [ ]:
all_streams, all_streams_datasources = perform_process_all_streams(streams=streams)

In [ ]:
all_streams_datasources['Epoc X Motion']


In [ ]:
# active_datasources = eeg_datasources
from pypho_timeline import SynchronizedPlotMode


active_datasources_dict = {k:v for k, v in all_streams_datasources.items() if v is not None}
active_datasources = list(active_datasources_dict.values())


# --- 4. Create the timeline widget and add EEG tracks ---
timeline = SimpleTimelineWidget(
    total_start_time=min([ds.total_df_start_end_times[0] for ds in active_datasources]),
    total_end_time=max([ds.total_df_start_end_times[1] for ds in active_datasources]),
    window_duration=10.0,
    window_start_time=min([ds.total_df_start_end_times[0] for ds in active_datasources]),
    add_example_tracks=False  # Don't add example tracks for XDF data
)

# Create plot widgets for each EEG stream and add tracks
for datasource in active_datasources:
    # Create a plot widget for this track
    track_widget, root_graphics, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
        name=datasource.custom_datasource_name,
        dockSize=(500, 80),
        dockAddLocationOpts=['bottom'],
        sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA
    )
    
    # Set the plot to show the full time range
    plot_item.setXRange(
        timeline.total_data_start_time, 
        timeline.total_data_end_time, 
        padding=0
    )
    plot_item.setYRange(0, 1, padding=0)
    plot_item.setLabel('bottom', 'Time', units='s')
    plot_item.setLabel('left', datasource.custom_datasource_name)
    plot_item.hideAxis('left')  # Hide Y-axis for cleaner look
    
    # Add the track to the plot
    timeline.add_track(
        datasource,
        name=datasource.custom_datasource_name,
        plot_item=plot_item
    )

timeline.setWindowTitle(f"pyPhoTimeline - EEG Modalities from XDF: {xdf_file_path.name}")
timeline.resize(1000, 800)
timeline.show()

print("\nTimeline widget created with EEG tracks from XDF:")
for ds in active_datasources:
    print(f"  - {ds.custom_datasource_name}, time: {ds.total_df_start_end_times}")

print("\nScroll on the timeline to see loaded EEG intervals for each stream.")
print("Close the window to exit.\n")


# 2026-03-18 - Smarter cross-stream post-processing

In [ ]:
## cross-stream post-processing
from pypho_timeline.rendering.datasources.specific.eeg import EEGSpectrogramTrackDatasource, EEGTrackDatasource

eeg_ds: EEGTrackDatasource = timeline.track_datasources['EEG_Epoc X']
eeg_spectogram_ds: EEGSpectrogramTrackDatasource = timeline.track_datasources['EEG_Spectrogram_Epoc X']


In [ ]:
# eeg_spectogram_ds.intervals_df
eeg_spectogram_ds.detailed_df